# Distancia de cada siniestro a la cámara más cercana
Objetivo: saber qué siniestros y qué víctimas quedaron a 300 m o menos de una cámara.
Las tablas guardan 2015-2025. Para responder la pregunta usamos 2019-2025.
Los años 2015-2018 solo se usan para comparar el antes y el después.

In [17]:
import pandas as pd

siniestros = pd.read_csv("../data/processed/siniestros.csv.gz")
victimas = pd.read_csv("../data/processed/victimas.csv.gz")
camaras = pd.read_csv("../data/processed/camaras.csv")
print(siniestros.shape, victimas.shape, camaras.shape)

(125154, 11) (198030, 13) (102, 8)


Distancia año por año

In [18]:
puntos = siniestros[["Codigo_Accidente", "AA_Acc", "Latitud", "Longitud"]]
cams = camaras[["id_camara", "latitud", "longitud"]]

partes = []

for anio in range(2015, 2026):
    del_anio = puntos[puntos["AA_Acc"] == anio]

    pares = del_anio.merge(cams, how="cross")

    pares["dy"] = (pares["Latitud"] - pares["latitud"]) * 111000
    pares["dx"] = (pares["Longitud"] - pares["longitud"]) * 111000

    pares["distancia_m"] = (pares["dx"] ** 2 + pares["dy"] ** 2) ** 0.5

    pares = pares.sort_values(["Codigo_Accidente", "distancia_m"])
    cercana = pares.drop_duplicates(subset="Codigo_Accidente", keep="first")

    partes.append(cercana[["Codigo_Accidente", "id_camara", "distancia_m"]])
    print(anio, len(del_anio), len(pares), len(cercana))

cercanas = pd.concat(partes, ignore_index=True)
print(cercanas.shape)

2015 10226 1043052 10226
2016 10330 1053660 10330
2017 10189 1039278 10189
2018 11895 1213290 11895
2019 12151 1239402 12151
2020 8820 899640 8820
2021 11467 1169634 11467
2022 12595 1284690 12595
2023 12999 1325898 12999
2024 12909 1316718 12909
2025 11573 1180446 11573
(125154, 3)


Cámara más cercana a cada siniestro

In [19]:
siniestros = siniestros.merge(cercanas, on="Codigo_Accidente", how="left", validate="1:1")

siniestros["cerca_300"] = siniestros["distancia_m"] <= 300
siniestros["cerca_500"] = siniestros["distancia_m"] <= 500

print(siniestros.shape)
print(siniestros["distancia_m"].isna().sum())

(125154, 15)
0


Pasarle esa información a cada víctima

In [20]:
victimas = victimas.merge(siniestros[["Codigo_Accidente", "id_camara", "distancia_m", "cerca_300", "cerca_500"]],
                          on="Codigo_Accidente", how="left", validate="m:1")
print(victimas.shape)
print(victimas["distancia_m"].isna().sum())

(198030, 17)
0


Guardar 

In [21]:
siniestros.to_csv("../data/processed/siniestros_camaras.csv.gz", index=False)
victimas.to_csv("../data/processed/victimas_camaras.csv.gz", index=False)

Resultado

In [22]:
periodo = victimas[victimas["AA_Acc"] >= 2019]
print("Víctimas 2019-2025:", len(periodo))

print("Víctimas a 300 m o menos:", round(periodo["cerca_300"].mean() * 100, 1), "%")
print("Víctimas a 500 m o menos:", round(periodo["cerca_500"].mean() * 100, 1), "%")
print("Distancia mediana a la cámara más cercana:", round(periodo["distancia_m"].median()), "m")

muertos = periodo[periodo["Gravedad_Indicador_30d"] == "MUERTO"]
print("Muertos 2019-2025:", len(muertos))
print("Muertos a 300 m o menos:", round(muertos["cerca_300"].mean() * 100, 1), "%")

Víctimas 2019-2025: 138178
Víctimas a 300 m o menos: 8.8 %
Víctimas a 500 m o menos: 17.6 %
Distancia mediana a la cámara más cercana: 1120 m
Muertos 2019-2025: 3504
Muertos a 300 m o menos: 10.7 %


## Decisiones
- Unimos siniestros y cámaras por coordenadas, medimos la distancia de cada siniestro a cada una de las cámaras y nos quedamos con la más cercana.
- Distancia en línea recta: pasamos los grados a metros (1 grado = 111.000 m en Bogotá) y usamos Pitágoras.
- Lo hicimos año por año porque todos con todos daba casi 13 millones de filas.
- Resultado: una fila por siniestro y una fila por víctima. Ninguna víctima quedó sin distancia.
- Guardamos todos los años (2015-2025). Para la pregunta filtramos 2019-2025.

## Primer hallazgo (2019-2025)
- Solo el 8,8 % de las víctimas y el 10,6 % de los muertos ocurrieron a 300 m o menos de una cámara. Esto no dice si las cámaras funcionan, dice dónde está la red de cámaras de hoy comparada con dónde han ocurrido las víctimas.